# Part A1 - Corpus Generation

In [1]:
"""Multilingual Indic Parallel Dataset Extraction Pipeline.
Source: yash9439/flores200 (Hugging Face)

Extracts and samples random entries from the dataset, according to user specified values for "NUMBER_OF_RECORDS", "LINES_PER_RECORD"
across English, Hindi, Malayalam, and Tamil languages into individual .txt files.
"""

import random
import re
import unicodedata
from typing import List
from datasets import load_dataset

#Dataset configuration
FLORES_REPO = "yash9439/flores200"
FLORES_SPLIT = "dev"

#Language column mappings in FLORES-200
LANGUAGE_CONFIGS = {
    "eng": "eng_Latn",  # English
    "hin": "hin_Deva",  # Hindi
    "mal": "mal_Mlym",  # Malayalam
    "tamil": "tam_Taml",  # Tamil
}

#Pipeline configuration defaults
random.seed(42)
NUMBER_OF_RECORDS = 30  # Total number of records (samples) to generate
LINES_PER_RECORD = 3  # Number of lines in each record

#Sanitizes text via standardizing Unicode to NFKC, and striping non-semantic invisible codepoints while preserving Devanagari & Dravidian ZWJ/ZWNJ.
def clean_indic_text(text):
    if not text:
        return ""

    #Unicode Normalization (Form KC)
    text = unicodedata.normalize("NFKC", text)

    #Remove zero-width spaces and BOMs
    text = text.replace("\ufeff", "").replace("\u200b", "").replace("\xa0", " ")

    #Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text).strip()
    return text

#Loading the FLORES-200 dataset split using Hugging Face datasets.
def load_flores_split(repo_name, split_name):
    dataset = load_dataset(repo_name, split=split_name)
    print(f"Dataset loaded successfully. Total sentences: {len(dataset)}\n")
    return dataset

#Generating unique random starting line numbers ensuring enough remaining lines for each record.
def generate_random_ln(total_dataset_size, num_records, num_lines):
    max_start_id = total_dataset_size - num_lines + 1

    if max_start_id < 1:
        raise ValueError(
            "Dataset is smaller than the requested number of lines per record."
        )

    if num_records > max_start_id:
        raise ValueError(
            f"Requested {num_records} records, but only {max_start_id} unique starting points exist."
        )

    # Randomly sample unique starting points
    ln = random.sample(range(1, max_start_id + 1), num_records)
    ln.sort()
    return ln

#Extracting N consecutive lines starting from the given line numbers
def extract_n_lines(dataset, lang_column, ln, num_lines):
    start_idx = ln - 1
    total_rows = len(dataset)

    if 0 <= start_idx < total_rows:
        end_idx = min(start_idx + num_lines, total_rows)
        lines = [
            clean_indic_text(dataset[i][lang_column])
            for i in range(start_idx, end_idx)
        ]
        return "\n".join(lines)

    return ""

#Saving records into a .txt file separated by record dividers
def save_list_to_file(data_list, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for idx, text in enumerate(data_list, 1):
            f.write(f"--- RECORD {idx} ---\n")
            f.write(text if text else "[RECORD OUT OF RANGE]")
            f.write("\n\n" + "=" * 50 + "\n\n")

    print(f" Saved: {filename} ({len(data_list)} records)")

#Sampling random line numbers and extracting "num_lines" for each language across "num_records", and writes individual .txt files
def process_flores_records(num_records = 5, num_lines = 10):
    dataset = load_flores_split(FLORES_REPO, FLORES_SPLIT)

    ln = generate_random_ln(len(dataset), num_records, num_lines)
    print(f"Generated {num_records} random starting line numbers: {ln}\n")

    eng_list = []
    hin_list = []
    mal_list = []
    tamil_list = []

    for idx, prid in enumerate(ln, 1):
        print(
            f"--> [Record {idx}/{num_records}] Extracting {num_lines} lines starting at PRID {prid}:"
        )

        eng_text = extract_n_lines(
            dataset, LANGUAGE_CONFIGS["eng"], prid, num_lines
        )
        hin_text = extract_n_lines(
            dataset, LANGUAGE_CONFIGS["hin"], prid, num_lines
        )
        mal_text = extract_n_lines(
            dataset, LANGUAGE_CONFIGS["mal"], prid, num_lines
        )
        tamil_text = extract_n_lines(
            dataset, LANGUAGE_CONFIGS["tamil"], prid, num_lines
        )

        eng_list.append(eng_text)
        hin_list.append(hin_text)
        mal_list.append(mal_text)
        tamil_list.append(tamil_text)

        print(
            f"    - English: {'Done' if eng_text else 'Failed'} | "
            f"Hindi: {'Done' if hin_text else 'Failed'} | "
            f"Malayalam: {'Done' if mal_text else 'Failed'} | "
            f"Tamil: {'Done' if tamil_text else 'Failed'}"
        )

    print("\nWriting language outputs to .txt files...")
    save_list_to_file(eng_list, "eng_output.txt")
    save_list_to_file(hin_list, "hin_output.txt")
    save_list_to_file(mal_list, "mal_output.txt")
    save_list_to_file(tamil_list, "tamil_output.txt")

    return eng_list, hin_list, mal_list, tamil_list


# Execution block
if __name__ == "__main__":
    # Specify the number of records and lines per record here
    eng_list, hin_list, mal_list, tamil_list = process_flores_records(
        num_records=NUMBER_OF_RECORDS, num_lines=LINES_PER_RECORD
    )

c:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded successfully. Total sentences: 997

Generated 30 random starting line numbers: [26, 28, 31, 33, 90, 96, 105, 115, 143, 204, 224, 229, 239, 251, 282, 433, 518, 559, 575, 605, 617, 655, 666, 693, 719, 734, 755, 759, 760, 914]

--> [Record 1/30] Extracting 3 lines starting at PRID 26:
    - English: Done | Hindi: Done | Malayalam: Done | Tamil: Done
--> [Record 2/30] Extracting 3 lines starting at PRID 28:
    - English: Done | Hindi: Done | Malayalam: Done | Tamil: Done
--> [Record 3/30] Extracting 3 lines starting at PRID 31:
    - English: Done | Hindi: Done | Malayalam: Done | Tamil: Done
--> [Record 4/30] Extracting 3 lines starting at PRID 33:
    - English: Done | Hindi: Done | Malayalam: Done | Tamil: Done
--> [Record 5/30] Extracting 3 lines starting at PRID 90:
    - English: Done | Hindi: Done | Malayalam: Done | Tamil: Done
--> [Record 6/30] Extracting 3 lines starting at PRID 96:
    - English: Done | Hindi: Done | Malayalam: Done | Tamil: Done
--> [Record 7/30

# Fertility Audit - Test on Above Corpus

This notebook runs the **corrected** `fertility.py`

The input for the relevant corpus files will be taken automatically from the reserved colab storage.


In [34]:
import sys
!pip install -q tiktoken transformers sentencepiece regex

## Section 2 — Corrected `fertility.py` logic

This is the same corrected script from `fertility.py` given in the submission repo, inlined here as functions so it can run interactively cell-by-cell. Every fix is commented with what was wrong in v0 and why the replacement is correct — see `partA/fertility.py` or the A2 audit writeup for the full isolated before/after evidence behind each one.

In [35]:
import unicodedata
import regex


def read_lines(path: str):
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            line = unicodedata.normalize("NFC", line)
            lines.append(line)
    return lines


def count_words_clean(line: str) -> int:
    """
    FIX (2): v0 used `line.split(" ")`, which splits only on a literal
    single space and produces empty-string entries for double spaces or
    irregular whitespace, inflating the word count. `.split()` with no
    argument splits on any whitespace run and discards empties.
    """
    return len(line.split())


def count_graphemes(line: str) -> int:
    return len(regex.findall(r"\X", line))


def preprocess_for_tokenization(line: str, lang: str) -> str:
    """
    FIX (1): v0 applied `.lower()` unconditionally. Hindi/Tamil/Malayalam
    have no case distinction, so `.lower()` is a no-op on them but rewrites
    nearly every English sentence -- asymmetric preprocessing before the
    two sides are ever compared. Fix: only lowercase English.
    """
    if lang == "eng":
        line = line.lower()
    return line


def analyze(lines, encode, lang: str):
    """
    FIX (3): v0 averaged per-line ratios (macro average), giving short,
    noisy lines equal weight to long ones. This pools tokens/words/graphemes
    across the whole corpus first, then divides once (micro average) --
    the honest corpus-level number.
    """
    total_tokens = 0
    total_words = 0
    total_graphemes = 0

    for raw_line in lines:
        line = preprocess_for_tokenization(raw_line, lang)
        tokens = encode(line)
        total_tokens += len(tokens)
        total_words += count_words_clean(line)
        total_graphemes += count_graphemes(line)

    fertility = total_tokens / total_words
    tok_per_grapheme = total_tokens / total_graphemes
    return fertility, tok_per_grapheme


def compression_ratio(lines, encode, lang: str) -> float:
    """
    FIX (4), conceptual bug: v0 called tokens/char "compression" -- backwards.
    Real compression is content-per-token (higher = better). This returns
    graphemes-per-token, the corrected direction, using the corrected
    grapheme-based character count from FIX (3).
    """
    total_tokens = 0
    total_graphemes = 0
    for raw_line in lines:
        line = preprocess_for_tokenization(raw_line, lang)
        tokens = encode(line)
        total_tokens += len(tokens)
        total_graphemes += count_graphemes(line)
    return total_graphemes / total_tokens


## Section 3 — Load your uploaded corpus files

In [36]:
CORPUS_FILES = {
    "eng": "eng_output.txt",
    "hin": "hin_output.txt",
    "mal": "mal_output.txt",
    "tam": "tamil_output.txt",
}

corpora = {lang: read_lines(path) for lang, path in CORPUS_FILES.items()}

for lang, lines in corpora.items():
    print(f"{lang}: {len(lines)} lines loaded" if lines else f"{lang}: 0 lines (file empty or missing)")


eng: 150 lines loaded
hin: 150 lines loaded
mal: 150 lines loaded
tam: 150 lines loaded


## Section 4 — Run the tokenizer(s)

Runs with `tiktoken gpt2` by default. Add or swap tokenizers by editing `TOKENIZERS` below — e.g. `"hf:xlm-roberta-base"` for a multilingual/Indic-aware comparison, per A3's requirement of at least two tokenizers.

In [37]:
import tiktoken

def load_tokenizer(spec: str):
    if spec.startswith("hf:"):
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(spec[3:])
        return lambda s: tok.encode(s, add_special_tokens=False)
    enc = tiktoken.get_encoding(spec)
    return enc.encode


TOKENIZERS = {
    "gpt2": load_tokenizer("gpt2"),
    "xlm-roberta-base": load_tokenizer("hf:xlm-roberta-base"),  # multilingual/Indic-aware, required for A3
}


In [38]:
import pandas as pd

results = []
for tok_name, encode in TOKENIZERS.items():
    for lang, lines in corpora.items():
        if not lines:
            continue
        fertility, tok_per_grapheme = analyze(lines, encode, lang)
        compression = compression_ratio(lines, encode, lang)
        results.append({
            "tokenizer": tok_name,
            "lang": lang,
            "n_lines": len(lines),
            "fertility_tok_per_word": round(fertility, 3),
            "tok_per_grapheme": round(tok_per_grapheme, 4),
            "compression_grapheme_per_tok": round(compression, 3),
        })

results_df = pd.DataFrame(results)
results_df.drop(columns = ["tok_per_grapheme"])


,tokenizer,lang,n_lines,fertility_tok_per_word,compression_grapheme_per_tok
0,gpt2,eng,150,1.297,5.049
1,gpt2,hin,150,7.251,0.551
2,gpt2,mal,150,23.757,0.255
3,gpt2,tam,150,22.108,0.294
4,xlm-roberta-base,eng,150,1.445,4.530
5,xlm-roberta-base,hin,150,1.496,2.672
6,xlm-roberta-base,mal,150,2.647,2.290
7,xlm-roberta-base,tam,150,2.412,2.696


# Part A2 - Isolation experiments

Run these against your own loaded corpus to get real before/after numbers for your writeup, rather than relying on the toy examples used during the original audit.

In [39]:
# ---- Evidence for FIX (1): .lower() asymmetry, using your own Hindi/English lines ----
enc = tiktoken.get_encoding("gpt2")

sample_eng = corpora["eng"][1] if corpora["eng"] else ""
sample_hin = corpora["hin"][1] if corpora["hin"] else ""

print("ENGLISH tokens -- with vs without unconditional lowercasing")
print("  original :", len(enc.encode(sample_eng)))
print("  lowered  :", len(enc.encode(sample_eng.lower())))

print("HINDI tokens -- with vs without unconditional lowercasing (expect no change)")
print("  original :", len(enc.encode(sample_hin)))
print("  lowered  :", len(enc.encode(sample_hin.lower())))


ENGLISH tokens -- with vs without unconditional lowercasing
  original : 24
  lowered  : 23
HINDI tokens -- with vs without unconditional lowercasing (expect no change)
  original : 136
  lowered  : 136


In [40]:
# ---- Evidence for FIX (2): naive split(" ") vs clean split() ----
injected = "The quick  brown fox jumps."  # double space injected on purpose

print("naive split(' ') word count :", len(injected.split(" ")))
print("clean split() word count    :", len(injected.split()))


naive split(' ') word count : 6
clean split() word count    : 5


In [41]:
# ---- Evidence for FIX (3): pooled average vs mean-of-ratios, using your own corpus ----
lang_to_test = "eng"
lines = corpora[lang_to_test]

if lines:
    ratios = [len(enc.encode(l)) / max(len(l.split()), 1) for l in lines]
    mean_of_ratios = sum(ratios) / len(ratios)

    total_tok = sum(len(enc.encode(l)) for l in lines)
    total_wrd = sum(len(l.split()) for l in lines)
    pooled = total_tok / total_wrd

    print(f"mean-of-ratios (v0, on {lang_to_test}) :", round(mean_of_ratios, 4))
    print(f"pooled (fixed, on {lang_to_test})      :", round(pooled, 4))
    print(f"relative difference: {abs(mean_of_ratios - pooled) / pooled * 100:.1f}%")
else:
    print(f"no lines loaded for '{lang_to_test}' -- upload the file and re-run Section 3")


mean-of-ratios (v0, on eng) : 1.5939
pooled (fixed, on eng)      : 1.2599
relative difference: 26.5%


In [42]:
# ---- Evidence for FIX (4): correction of compression metric ----
def analyze_comp(lines, encode, lang: str):
    total_tokens = 0
    total_chars = 0

    for raw_line in lines:
        line = preprocess_for_tokenization(raw_line, lang)
        tokens = encode(line)
        total_tokens += len(tokens)
        total_chars += len(line)

    char_per_token = total_chars / total_tokens
    token_per_char = total_tokens / total_chars
    return char_per_token, token_per_char
results1 =[]
for tok_name, encode in TOKENIZERS.items():
    for lang, lines in corpora.items():
        if not lines:
            continue
        char_per_token, token_per_char = analyze_comp(lines, encode, lang)
        compression = compression_ratio(lines, encode, lang)
        results1.append({
            "tokenizer": tok_name,
            "lang": lang,
            "char_per_token": round(char_per_token, 3),
            "token_per_char": round(token_per_char, 4)
        })
results1_df = pd.DataFrame(results1)
results1_df

,tokenizer,lang,char_per_token,token_per_char
0,gpt2,eng,5.049,0.1981
1,gpt2,hin,0.771,1.2975
2,gpt2,mal,0.425,2.3543
3,gpt2,tam,0.424,2.3606
4,xlm-roberta-base,eng,4.530,0.2207
5,xlm-roberta-base,hin,3.736,0.2677
6,xlm-roberta-base,mal,3.812,0.2623
7,xlm-roberta-base,tam,3.883,0.2575


# Part A3: corrected cross-language comparison, four denominators

The comparison with **two tokenizers** (`gpt2` and `xlm-roberta-base`, both loaded in Section 4) and **at least two denominators** — here, all four candidates are computed so you can compare them side by side: per whitespace word, per grapheme cluster, per UTF-8 byte, and per parallel sentence.



In [43]:
def count_utf8_bytes(line: str) -> int:
    return len(line.encode("utf-8"))


def alignment_check(corpora: dict) -> bool:
    """Line counts matching is necessary but not sufficient for true
    parallel alignment"""
    lengths = {lang: len(lines) for lang, lines in corpora.items()}
    all_equal = len(set(lengths.values())) == 1
    print(f"line counts per language: {lengths}")
    print(f"all equal (necessary for parallel alignment): {all_equal}")
    if not all_equal:
        print("-> corpora are NOT the same length. Per-parallel-sentence "
              "denominator will be skipped below; word/grapheme/byte "
              "denominators are still valid since they don't require alignment.")
    return all_equal


IS_PARALLEL = alignment_check(corpora)


line counts per language: {'eng': 150, 'hin': 150, 'mal': 150, 'tam': 150}
all equal (necessary for parallel alignment): True


In [44]:
def tokens_per_byte(lines, encode, lang: str) -> float:
    total_tokens = 0
    total_bytes = 0
    for raw_line in lines:
        line = preprocess_for_tokenization(raw_line, lang)
        total_tokens += len(encode(line))
        total_bytes += count_utf8_bytes(line)
    return total_tokens / total_bytes


def tokens_per_parallel_sentence(lines, encode, lang: str) -> float:
    """Only call this if IS_PARALLEL is True"""
    total_tokens = 0
    for raw_line in lines:
        line = preprocess_for_tokenization(raw_line, lang)
        total_tokens += len(encode(line))
    return total_tokens / len(lines)


a3_results = []
for tok_name, encode in TOKENIZERS.items():
    for lang, lines in corpora.items():
        if not lines:
            continue
        fertility, tok_per_grapheme = analyze(lines, encode, lang)
        tok_per_byte = tokens_per_byte(lines, encode, lang)
        row = {
            "tokenizer": tok_name,
            "lang": lang,
            "tok_per_word": round(fertility, 3),
            "tok_per_grapheme": round(tok_per_grapheme, 4),
            "tok_per_byte": round(tok_per_byte, 4),
        }
        if IS_PARALLEL:
            row["tok_per_parallel_sentence"] = round(tokens_per_parallel_sentence(lines, encode, lang), 2)
        a3_results.append(row)

a3_df = pd.DataFrame(a3_results)
a3_df


,tokenizer,lang,tok_per_word,tok_per_grapheme,tok_per_byte,tok_per_parallel_sentence
0,gpt2,eng,1.297,0.1981,0.1980,16.83
1,gpt2,hin,7.251,1.8140,0.5608,110.85
2,gpt2,mal,23.757,3.9194,0.9441,219.51
3,gpt2,tam,22.108,3.4001,0.9477,232.73
4,xlm-roberta-base,eng,1.445,0.2207,0.2207,18.76
5,xlm-roberta-base,hin,1.496,0.3742,0.1157,22.87
6,xlm-roberta-base,mal,2.647,0.4367,0.1052,24.46
7,xlm-roberta-base,tam,2.412,0.3709,0.1034,25.39
